# 04 — Feature Engineering: M5 Walmart Demand Intelligence

**Goal:** Build the feature matrix that XGBoost consumes in notebook 05.
Every feature below has a direct EDA justification. No speculative features.

**Inputs:** Raw CSVs (`sales_train_validation.csv`, `calendar.csv`, `sell_prices.csv`)
**Output:** `../data/processed/features_train.parquet`, `features_val.parquet`
**Granularity:** `item_id × store_id × date` (daily)
**Target:** `log1p(units_sold)` — EDA confirmed skewness=11.89; log-transform stabilises variance for tree models

> **Scope:** Features are built across all 30,490 product-store series.
> EDA Section 22 confirmed a mean longest zero streak of 430 days per series —
> the majority of zeros reflect structural demand censoring (product unavailable)
> rather than genuine zero demand. Lag features are computed with gap-aware logic:
> any lag window spanning a zero streak longer than 28 consecutive days is nulled out.
> XGBoost handles nulls natively. This is more realistic than restricting to the
> 2,469 complete series and reflects how a production demand system must operate.

## 1. Imports and Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

sns.set_style('whitegrid')
warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (14, 5)
np.random.seed(42)

RAW_DIR       = '../data/raw'
PROCESSED_DIR = '../data/processed'
REP_SERIES    = 'FOODS_3_163_CA_3_validation'  # used for spot-checks only

# Lag windows — driven by M5 evaluation horizon (28-day forecast)
LAG_DAYS     = [1, 7, 14, 28]
ROLLING_DAYS = [7, 28]

# Gap threshold — lags spanning a zero streak longer than this are nulled out.
# 28 days = one full forecast horizon. A product absent this long is structurally
# unavailable, not intermittently slow. Matches EDA Section 22 finding.
GAP_THRESHOLD = 28

## 2. Load Raw Data & Rebuild Long-Format Frame

Same pipeline as the EDA notebook. All 30,490 product-store series are
melted from wide to long format — no pre-filtering. Categorical dtypes
are applied before the calendar join to cut memory ~70%, keeping the
58M row frame manageable. Jan 2011 is dropped (only 3 days captured),
consistent with all baseline notebooks.

In [ ]:
print('Loading raw files...')
sales_wide = pd.read_csv(f'{RAW_DIR}/sales_train_validation.csv')
calendar   = pd.read_csv(f'{RAW_DIR}/calendar.csv')
prices     = pd.read_csv(f'{RAW_DIR}/sell_prices.csv')

print(f'  sales_train_validation: {sales_wide.shape}')
print(f'  calendar:               {calendar.shape}')
print(f'  sell_prices:            {prices.shape}')
print()

# Melt ALL 30,490 series — no pre-filter
# Gap-aware lag logic handles zero streaks downstream
id_cols  = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id']
day_cols = [c for c in sales_wide.columns if c.startswith('d_')]

df = sales_wide.melt(
    id_vars=id_cols,
    value_vars=day_cols,
    var_name='d',
    value_name='units_sold'
)
df['units_sold'] = df['units_sold'].astype('int16')

# Cast to categorical BEFORE calendar join — cuts memory ~70%
for col in ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd']:
    df[col] = df[col].astype('category')

# Join calendar — slim to only needed columns
cal_cols      = ['d', 'date', 'wm_yr_wk', 'weekday', 'month', 'year',
                 'event_name_1', 'event_type_1', 'snap_CA', 'snap_TX', 'snap_WI']
calendar_slim = calendar[calendar['d'].isin(df['d'].cat.categories)][cal_cols].copy()

df = df.merge(calendar_slim, on='d', how='left')
df['date'] = pd.to_datetime(df['date'])

# Drop incomplete Jan 2011 — only 3 days captured, consistent with baselines
df = df[df['date'] >= '2011-02-01'].reset_index(drop=True)

print(f'Series:     {df["id"].nunique():,}')
print(f'Shape:      {df.shape}')
print(f'Memory:     {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'Date range: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Expected rows: {30490 * len(day_cols):,}  (30,490 series × {len(day_cols)} days)')

## 3. Join Prices

Prices are weekly per `store_id × item_id`. Join on `wm_yr_wk`.
Missing prices filled with forward-fill within each series — a product
that existed last week at a given price almost certainly has the same
price this week if no new record exists. Any remaining nulls (product
not yet priced) are filled with 0.

In [ ]:
# Cast remaining string columns — brings memory down from 14 GB
for col in ['weekday', 'event_name_1', 'event_type_1']:
    df[col] = df[col].astype('category')

# Sort by series and date — required for all lag/rolling operations
df = df.sort_values(['id', 'date']).reset_index(drop=True)

# Join prices at item × store × week level
df = df.merge(
    prices[['store_id', 'item_id', 'wm_yr_wk', 'sell_price']],
    on=['store_id', 'item_id', 'wm_yr_wk'],
    how='left'
)

# Forward-fill within each series — a product keeps its last known price
# until a new price record exists. Fill any leading nulls (not yet priced) with 0.
df['sell_price'] = (
    df.groupby('id', observed=True)['sell_price']
    .transform(lambda x: x.ffill().fillna(0))
    .astype('float32')
)

print(f'Null sell_price after fill: {df["sell_price"].isna().sum()}')
print(f'Shape after price join:     {df.shape}')
print(f'Memory after cast + join:   {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print(f'sell_price stats:')
print(df['sell_price'].describe().round(3))

## 4. Temporal Features

Calendar decomposition. XGBoost cannot extract date structure from a
raw timestamp — these must be explicit columns. `is_month_start` and
`is_month_end` capture the demand pulse seen at pay-period boundaries.

In [ ]:
df['day_of_week']    = df['date'].dt.dayofweek.astype('int8')   # 0=Mon, 6=Sun
df['day_of_month']   = df['date'].dt.day.astype('int8')
df['week_of_year']   = df['date'].dt.isocalendar().week.astype('int16')
df['month_num']      = df['date'].dt.month.astype('int8')
df['is_weekend']     = (df['day_of_week'] >= 5).astype('int8')
df['is_month_start'] = df['date'].dt.is_month_start.astype('int8')
df['is_month_end']   = df['date'].dt.is_month_end.astype('int8')

print('Temporal features added.')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print()
print(df[['day_of_week','day_of_month','week_of_year',
          'month_num','is_weekend','is_month_start','is_month_end']].describe().round(2))

Calendar decomposition — XGBoost cannot extract date structure from a
raw timestamp. Seven explicit features covering day, week, and month
cycles visible in the EDA seasonal decomposition.

- `day_of_week` mean=3.0, uniform 0–6 ✓
- `is_weekend` rate=29% — matches expected 2/7 days ✓
- `is_month_start` / `is_month_end` rate=3% each — ~1 day per month ✓
- Memory ticked up to **14.40 GB** — expected, 7 new int8/int16 columns
  across 58M rows.

## 5. Event & SNAP Features

**Holiday encoding:**
EDA Section 11 showed same-day demand is *negative* for Christmas (-100%)
and Thanksgiving (-41%) — demand is displaced into the days before.
Lead windows capture this. SuperBowl/LaborDay show positive same-day
effects — no lead needed.

**SNAP encoding:**
EDA Section 12 confirmed SNAP effect is state-specific (+10.3% CA,
+17.2% TX, +32.5% WI for FOODS). A single global flag would lose this.
We match the correct state flag to each row via `state_id`.

In [ ]:
# ── Binary event flags ────────────────────────────────────────────────────
df['is_event']          = df['event_name_1'].notna().astype('int8')
df['is_closed_holiday'] = df['event_name_1'].isin(
    ['Christmas', 'Thanksgiving']
).astype('int8')

# ── Days to next closed holiday ───────────────────────────────────────────
# EDA Section 11: Christmas (-100%) and Thanksgiving (-40.8%) suppress
# same-day observed demand — stores close or traffic shifts to lead days.
# Signal lives in the 3 days BEFORE, not day-of. Encode the distance.

closed_holiday_dates = pd.to_datetime(
    calendar[calendar['event_name_1'].isin(
        ['Christmas', 'Thanksgiving'])]['date'].unique()
)

all_dates = df['date'].unique()
date_to_days = {}

for d in all_dates:
    deltas = (closed_holiday_dates - d).days
    forward = deltas[deltas >= 0]
    date_to_days[d] = int(forward.min()) if len(forward) > 0 else GAP_THRESHOLD + 1

df['days_to_closed_holiday'] = (
    df['date'].map(date_to_days)
    .clip(upper=GAP_THRESHOLD + 1)
    .astype('int8')
)
df['is_pre_closed_holiday'] = (df['days_to_closed_holiday'] <= 3).astype('int8')

print('Event flags added.')
print(f'is_event rate:              {df["is_event"].mean()*100:.1f}%')
print(f'is_closed_holiday rate:     {df["is_closed_holiday"].mean()*100:.1f}%')
print(f'is_pre_closed_holiday rate: {df["is_pre_closed_holiday"].mean()*100:.1f}%')
print()
print('days_to_closed_holiday value counts (top 10):')
print(df['days_to_closed_holiday'].value_counts().sort_index().head(10))

In [ ]:
state_id_str = df['state_id'].astype(str)

df['is_snap'] = np.where(state_id_str == 'CA', df['snap_CA'],
                np.where(state_id_str == 'TX', df['snap_TX'],
                                               df['snap_WI'])).astype('int8')

df.drop(columns=['snap_CA', 'snap_TX', 'snap_WI'], inplace=True)

print('SNAP feature added.')
print(f'is_snap rate overall:  {df["is_snap"].mean()*100:.1f}%')
print()
print('is_snap rate by state:')
print(df.groupby('state_id', observed=True)['is_snap'].mean().mul(100).round(1))
print()
print('Cross-check — should match exactly:')
df_check = df.merge(
    calendar_slim[['d', 'snap_CA', 'snap_TX', 'snap_WI']],
    on='d', how='left'
)
for state, snap_col in [('CA','snap_CA'), ('TX','snap_TX'), ('WI','snap_WI')]:
    raw  = df_check[df_check['state_id']==state][snap_col].mean() * 100
    enc  = df[df['state_id']==state]['is_snap'].mean() * 100
    match = '✓' if abs(raw - enc) < 0.01 else '✗'
    print(f'  {state}: raw {snap_col}={raw:.1f}%  is_snap={enc:.1f}%  {match}')

print()
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')

**Holiday encoding:**
EDA Section 11 showed same-day observed demand is negative for Christmas
(-100%) and Thanksgiving (-40.8%) — stores close, demand shifts into lead
days. `days_to_closed_holiday` encodes distance to the next closed holiday
(capped at GAP_THRESHOLD + 1 = outside window). `is_pre_closed_holiday`
flags the 3-day pre-closure window where demand actually lives.

- `is_event` rate: 8.1% — matches calendar structure ✓
- `is_closed_holiday` rate: 0.5% — ~10 days/year × 5 years ✓
- `is_pre_closed_holiday` rate: 2.1% — 3 lead days × ~10 holidays ✓
- `days_to_closed_holiday` uniform at 304,900 per value — exactly
  30,490 series × 10 holiday occurrences ✓

**SNAP encoding:**
EDA Section 12 confirmed SNAP uplift is state-specific (+10.3% CA,
+17.2% TX, +32.5% WI for FOODS). Each row gets its own state's SNAP
flag — a CA store sees CA's SNAP calendar, not TX or WI's.
All three states show ~33% SNAP days — correct, the M5 calendar
distributes benefits on roughly 1 in 3 days per state independently.
Cross-check confirms `is_snap` matches raw state snap columns exactly ✓

## 6. Price Features

EDA Section 18: price elasticity signal is **product-store level only**
(r=0.553 for drops at FOODS_3_383 × CA_3). Department-level price features
have r<0.13 — excluded.

EDA Section 17: asymmetric elasticity confirmed — drops and increases
behave differently. Signed `price_change_pct` preserves this. Separate
`price_drop` and `price_increase` flags let XGBoost learn the asymmetry
independently.

All computed within `item_id × store_id` groups. Never aggregated above that.

In [ ]:
# ── Price features ────────────────────────────────────────────────────────
# All computed at item_id × store_id level only.
# EDA Section 18: department-level price features have r<0.13 — excluded.
# EDA Section 18: asymmetric elasticity confirmed — drops and increases
# behave differently. Signed price_change_pct preserves direction.

grp_price = df.groupby('id', observed=True)['sell_price']

# Lagged price — price 7 days ago
df['price_lag_7'] = grp_price.shift(7).astype('float32')

# Raw price change % — preserved at full range for anomaly detection (nb 05)
df['price_change_pct_raw'] = (
    (df['sell_price'] - df['price_lag_7'])
    / df['price_lag_7'].replace(0, np.nan) * 100
).astype('float32')

# Capped version for modeling — ±200% covers all legitimate price moves
# 343 rows exceed this (0.0006% of data) — M5 source anomalies confirmed
# above (e.g. $0.10 → $3.28, likely unit price vs pack price errors)
df['price_change_pct'] = df['price_change_pct_raw'].clip(
    lower=-100, upper=200
).fillna(0).astype('float32')

# Direction flags — asymmetric elasticity means drops and increases
# must be encoded separately (EDA Section 18: r=0.553 drops vs r=0.18 increases)
df['price_drop']     = (df['price_change_pct'] < 0).astype('int8')
df['price_increase'] = (df['price_change_pct'] > 0).astype('int8')

# Rolling 28-day average price — medium-term price baseline
# shift(1) prevents today's price leaking into the rolling window
df['price_rolling_28'] = (
    grp_price
    .transform(lambda x: x.shift(1).rolling(28, min_periods=1).mean())
    .astype('float32')
)

# Price relative to 28-day baseline — how discounted is today
# Values < 1.0 = below recent baseline (potential demand driver)
# Null where no price history exists — XGBoost handles natively
df['price_rel_28'] = (
    df['sell_price'] / df['price_rolling_28'].replace(0, np.nan)
).astype('float32')

# Null out price features on unpriced rows (sell_price=0)
# These are structural gaps — product not yet on shelf, not a real $0 price
no_price = df['sell_price'] == 0
for col in ['price_lag_7', 'price_change_pct', 'price_change_pct_raw',
            'price_drop', 'price_increase', 'price_rolling_28', 'price_rel_28']:
    df.loc[no_price, col] = np.nan

print('Price features added.')
print(f'Rows with no valid price (sell_price=0): {no_price.sum():,} ({no_price.mean()*100:.1f}%)')
print(f'Anomalous price changes preserved in price_change_pct_raw: {(df["price_change_pct_raw"].abs() > 200).sum():,} rows')
print()
print(df[['sell_price','price_change_pct','price_change_pct_raw',
          'price_drop','price_increase','price_rel_28']].describe().round(3))

All features computed at `item_id × store_id` level only.
EDA Section 18: department-level price correlations r<0.13 — no signal
at aggregated levels, excluded entirely.

- **21.0% of rows have no valid price** (`sell_price=0`) — product not
  yet on shelf. Price features nulled on these rows; XGBoost handles
  natively via surrogate splits.
- **`price_change_pct`** capped at ±200% for modeling. 3,937 rows exceed
  this threshold — confirmed M5 source anomalies (e.g. $0.10 → $3.28,
  likely unit vs pack price inconsistencies). Raw values preserved in
  `price_change_pct_raw` for anomaly detection in notebook 05.
- **`price_drop` / `price_increase`** encoded separately — EDA Section 18
  confirmed asymmetric elasticity: r=0.553 for drops vs r=0.180 for
  increases at product-store level.
- **`price_rel_28`** mean=1.033 — products are on average 3.3% above
  their 28-day price baseline. Values below 1.0 signal active discounting,
  the strongest observed demand driver in the EDA.

## 7. Lag & Rolling Features

The strongest predictor of near-term demand is recent demand history.
SARIMA confirmed AR(1) is significant (p≈0.000) — this translates directly
to `lag_1`, `lag_7` etc. in the tabular feature space.

All features are computed across all 30,490 series. Gap-aware nulling
prevents lag windows from spanning structural zero streaks (mean=430 days
per series confirmed in EDA Section 22) — a lag that reaches back into a
demand censoring period pulls a supply-side zero into a predictive position
where it will be misread as genuine demand history. XGBoost handles the
resulting nulls natively via surrogate splits.

Rolling std captures demand volatility — a volatile series needs wider
safety stock. This feeds directly into the inventory decision layer.

In [ ]:
grp_sales = df.groupby('id', observed=True)['units_sold']

# ── Compute lag features ──────────────────────────────────────────────────
for lag in LAG_DAYS:
    df[f'lag_{lag}'] = grp_sales.shift(lag).astype('float32')

# ── Rolling features — shift(1) prevents today leaking into window ────────
for window in ROLLING_DAYS:
    df[f'rolling_mean_{window}'] = (
        grp_sales
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype('float32')
    )

df['rolling_std_7'] = (
    grp_sales
    .transform(lambda x: x.shift(1).rolling(7, min_periods=2).std())
    .fillna(0)
    .astype('float32')
)

# ── Gap-aware nulling ─────────────────────────────────────────────────────
# EDA Section 22: mean longest zero streak = 430 days. Zeros in these
# windows reflect structural demand censoring (product unavailable),
# not genuine demand history. Feeding them into lag positions would
# systematically bias predictions downward for products returning
# from stockouts or lifecycle gaps.
#
# Rule: if the product has had zero sales for every one of the last
# GAP_THRESHOLD (28) days, it is in a structural gap. Null all lag
# and rolling features on that row — XGBoost handles nulls natively
# via surrogate splits.

gap_mask = (
    grp_sales
    .transform(lambda x: x.shift(1).rolling(GAP_THRESHOLD, min_periods=GAP_THRESHOLD).max())
    == 0
)

lag_cols = (
    [f'lag_{l}'           for l in LAG_DAYS] +
    [f'rolling_mean_{w}'  for w in ROLLING_DAYS] +
    ['rolling_std_7']
)

for col in lag_cols:
    df.loc[gap_mask, col] = np.nan

print('Lag, rolling, and gap-aware features added.')
print(f'Rows identified as structural gaps: {gap_mask.sum():,} ({gap_mask.mean()*100:.1f}%)')
print()
print('Null counts per feature (gaps + natural edge nulls at series start):')
print(df[lag_cols].isna().sum().to_string())
print()
print(df[lag_cols].describe().round(3))

- **29.6% of rows flagged as structural gaps** — consistent with EDA
  finding that 44.1% of series have zero streaks exceeding 365 days.
- **Null counts increase with lag distance** — lag_1: 17.3M nulls,
  lag_28: 18.1M. Expected: longer lookbacks span more gap boundaries.
- **Lag max = 763** — matches EDA Section 22 confirmed single-day
  maximum ✓
- Rolling std mean=1.28 feeds directly into the safety stock formula
  in the Streamlit app layer: `safety_stock = z × rolling_std_7 × √lead_time`

## 8. Hierarchical Features

EDA Section 13: departments are correlated (0.64–0.96) but not perfectly —
department-level aggregates add signal beyond individual series.
EDA Section 14: stores vary 3x in revenue — store-level demand context
helps XGBoost separate signal from noise at the individual series level.

These are **rolling aggregates lagged by 1 day** to prevent leakage.
Today's store-level demand cannot be used to predict today's item-level demand.

In [ ]:
# ── Store-level rolling demand ────────────────────────────────────────────
store_daily = (
    df.groupby(['store_id', 'date'], observed=True)['units_sold']
    .sum().reset_index()
    .rename(columns={'units_sold': 'store_daily_units'})
    .sort_values(['store_id', 'date'])
)
for window in [7, 28]:
    store_daily[f'store_rolling_{window}'] = (
        store_daily.groupby('store_id', observed=True)['store_daily_units']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype('float32')
    )
df = df.merge(
    store_daily[['store_id', 'date', 'store_rolling_7', 'store_rolling_28']],
    on=['store_id', 'date'], how='left'
)

# ── Department-level rolling demand ───────────────────────────────────────
dept_daily = (
    df.groupby(['dept_id', 'store_id', 'date'], observed=True)['units_sold']
    .sum().reset_index()
    .rename(columns={'units_sold': 'dept_daily_units'})
    .sort_values(['dept_id', 'store_id', 'date'])
)
for window in [7, 28]:
    dept_daily[f'dept_rolling_{window}'] = (
        dept_daily.groupby(['dept_id', 'store_id'], observed=True)['dept_daily_units']
        .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
        .astype('float32')
    )
df = df.merge(
    dept_daily[['dept_id', 'store_id', 'date', 'dept_rolling_7', 'dept_rolling_28']],
    on=['dept_id', 'store_id', 'date'], how='left'
)

print('Hierarchical features added.')
print(df[['store_rolling_7', 'store_rolling_28',
          'dept_rolling_7',  'dept_rolling_28']].describe().round(2))

Store and department rolling demand context — allows XGBoost to distinguish
item-specific weakness from broader store or department softness.
Two windows: 7-day (short-term momentum) and 28-day (forecast horizon baseline).
All lagged 1 day to prevent leakage. Item-level rolling already captured
in Section 7 at `id` granularity.

- `store_rolling_7` max=7,673 vs `store_rolling_28` max=7,011 — short-term
  window captures demand spikes smoothed out over 28 days ✓
- `dept_rolling_7` min=0.14 — sparse departments can have near-zero
  short-term demand, stabilises over the 28-day window ✓

## 9. Encode Categorical ID Features

XGBoost natively handles integer-encoded categoricals. Label encoding
is sufficient — no one-hot needed. Tree splits on integer codes are
equivalent to one-hot for unordered categoricals in gradient boosting.

We encode `store_id`, `item_id`, `dept_id`, `cat_id`, `state_id` and
`weekday`. The `id` column (full product-store key) is retained as a
string for filtering/lookup but not used as a model feature.

In [ ]:
from sklearn.preprocessing import LabelEncoder
import pickle

# Drop join keys and duplicate calendar columns not used as features
df.drop(columns=['d', 'wm_yr_wk', 'month', 'year', 'event_type_1'], inplace=True)

CAT_COLS = ['store_id', 'item_id', 'dept_id', 'cat_id', 'state_id', 'weekday']

label_encoders = {}
for col in CAT_COLS:
    le = LabelEncoder()
    df[col] = df[col].astype(str)
    df[f'{col}_enc'] = le.fit_transform(df[col]).astype('int16')
    label_encoders[col] = le
    print(f'{col}: {le.classes_.shape[0]} unique values → int16')

# Re-cast back to category — astype(str) for LabelEncoder inflates memory
for col in CAT_COLS:
    df[col] = df[col].astype('category')

os.makedirs(PROCESSED_DIR, exist_ok=True)
with open(f'{PROCESSED_DIR}/label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)

print()
print(f'Shape after encoding: {df.shape}')
print(f'Memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB')
print('\nLabel encoders saved.')

GBoost requires integer-encoded categoricals. Label encoding is sufficient —
no one-hot needed. Tree splits on integer codes are equivalent to one-hot
for unordered categoricals in gradient boosting.

`d`, `wm_yr_wk`, `month`, `year`, `event_type_1` dropped — join keys and
calendar duplicates not used as model features.

- 6 categoricals encoded: `store_id` (10), `item_id` (3,049), `dept_id` (7),
  `cat_id` (3), `state_id` (3), `weekday` (7)
- Re-cast to category dtype after encoding — `astype(str)` for LabelEncoder
  inflated memory from 17.95 GB to 26.64 GB, re-cast recovers to 7.52 GB ✓
- Label encoders saved to `label_encoders.pkl` for inverse-transform
  in the Streamlit app layer

## 10. Build Target Variable

`log1p(units_sold)` — standard for count targets with heavy right skew.
EDA Section 22: skewness=11.89 on non-zero days; max=763 units.
Log-transform pulls the distribution toward normality and prevents
XGBoost from over-weighting the rare high-volume days.

`log1p` handles zeros correctly: `log1p(0) = 0`, so zero-sales days
stay at zero in target space. Back-transform with `np.expm1` at inference.

In [ ]:
df['target'] = np.log1p(df['units_sold']).astype('float32')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['units_sold'].clip(upper=20), bins=50, color='steelblue', edgecolor='none')
axes[0].set_title('units_sold (clipped at 20 for display)')
axes[0].set_xlabel('Units')

axes[1].hist(df['target'], bins=50, color='steelblue', edgecolor='none')
axes[1].set_title('log1p(units_sold) — target')
axes[1].set_xlabel('log1p(units)')

plt.suptitle('Target Variable: Raw vs Log-Transformed', fontsize=12)
plt.tight_layout()
plt.show()

print(f'Raw skewness:    {df["units_sold"].skew():.2f}')
print(f'Target skewness: {df["target"].skew():.2f}')

`log1p(units_sold)` — standard for count targets with heavy right skew.
`log1p(0) = 0` so zero-sales days stay at zero in target space.
Back-transform with `np.expm1` at inference.

- Raw skewness: 17.18 — higher than EDA's 11.89 because all 30,490
  series are included, adding more sparse and intermittent demand ✓
- Target skewness: 1.93 after transform — tractable for XGBoost ✓

## 11. Walk-Forward Train / Validation Split

XGBoost must be evaluated via **walk-forward validation** — no shuffling,
no random splits. Using future data to train on past data is leakage.

Split mirrors the statistical baselines for direct comparison:
- **Train:** up to Jan 31 2015 (same 48-month window)
- **Validation:** Feb 2015 → Jan 2016 (same 12-month held-out period)
- **Rows with null lags** (first 28 days per series) are dropped from train
  only — we cannot train on rows with missing lag features.

The validation set is touched exactly once — during final model evaluation
in notebook 05. It is constructed here for reference and saved with the
feature matrix.

In [ ]:
TRAIN_END = '2015-01-31'
VAL_START = '2015-02-01'
VAL_END   = '2016-01-31'

train_mask = df['date'] <= TRAIN_END
val_mask   = (df['date'] >= VAL_START) & (df['date'] <= VAL_END)

# Drop rows where ALL lag features are null — no signal to train on
# Rows where SOME lags are null (gap rows) are kept — XGBoost handles via surrogate splits
lag_cols_all = [f'lag_{l}' for l in LAG_DAYS]
has_any_lag  = df[lag_cols_all].notna().any(axis=1)

train_df = df[train_mask & has_any_lag].copy()
val_df   = df[val_mask].copy()

print(f'Train: {len(train_df):,} rows  ({train_df["date"].min().date()} → {train_df["date"].max().date()})')
print(f'Val:   {len(val_df):,} rows    ({val_df["date"].min().date()} → {val_df["date"].max().date()})')
print()
print(f'Train zero % (units_sold): {(train_df["units_sold"] == 0).mean()*100:.1f}%')
print(f'Val zero %   (units_sold): {(val_df["units_sold"] == 0).mean()*100:.1f}%')
print()
print('Null counts on train_df (spot check):')
check_cols = ['lag_1', 'sell_price', 'price_change_pct', 'rolling_mean_7', 'store_rolling_7']
print(train_df[check_cols].isna().sum().to_string())

## 12. Feature Summary & Validation

Final check before saving: confirm feature completeness, null rates,
and that no future data leaked into any feature.

In [ ]:
# ── Define exact feature set — reference for notebooks 05, 06, and app layer ──
FEATURE_COLS = [
    # Temporal (7)
    'day_of_week', 'day_of_month', 'week_of_year', 'month_num',
    'is_weekend', 'is_month_start', 'is_month_end',
    # Event / SNAP (5)
    'is_event', 'is_closed_holiday', 'days_to_closed_holiday',
    'is_pre_closed_holiday', 'is_snap',
    # Price (5) — price_change_pct_raw excluded (anomaly detection only, not a model feature)
    'sell_price', 'price_change_pct', 'price_drop',
    'price_increase', 'price_rel_28',
    # Lag / Rolling (7)
    'lag_1', 'lag_7', 'lag_14', 'lag_28',
    'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7',
    # Hierarchical (4)
    'store_rolling_7', 'store_rolling_28',
    'dept_rolling_7',  'dept_rolling_28',
    # Encoded categoricals (6)
    'store_id_enc', 'item_id_enc', 'dept_id_enc',
    'cat_id_enc', 'state_id_enc', 'weekday_enc',
]

TARGET_COL = 'target'
META_COLS  = ['id', 'date', 'units_sold', 'price_change_pct_raw']
# price_change_pct_raw kept in meta — passed to notebook 07 for anomaly detection

print(f'Total features: {len(FEATURE_COLS)}')
print()

# ── Null check on training set ────────────────────────────────────────────
# Nulls are expected in lag/price features — gap rows and unpriced products
# XGBoost handles these natively. Flag here for awareness, not to resolve.
null_summary = train_df[FEATURE_COLS].isna().sum()
null_summary = null_summary[null_summary > 0]
print('Null counts per feature (training set):')
print(null_summary.to_string())
print()

# ── Leakage check ─────────────────────────────────────────────────────────
# lag_1 must correlate with target but never = 1.0
# If it were 1.0 today's sales would be perfectly predicted by yesterday — leakage
lag1_corr = train_df['lag_1'].corr(train_df['target'])
print(f'lag_1 ↔ target correlation: {lag1_corr:.3f}  (expected: moderate positive, never 1.0)')
print()

# ── Feature group summary ─────────────────────────────────────────────────
groups = {
    'Temporal (7)':       ['day_of_week','day_of_month','week_of_year','month_num',
                            'is_weekend','is_month_start','is_month_end'],
    'Event/SNAP (5)':     ['is_event','is_closed_holiday','days_to_closed_holiday',
                            'is_pre_closed_holiday','is_snap'],
    'Price (5)':          ['sell_price','price_change_pct','price_drop',
                            'price_increase','price_rel_28'],
    'Lag/Rolling (7)':    ['lag_1','lag_7','lag_14','lag_28',
                            'rolling_mean_7','rolling_mean_28','rolling_std_7'],
    'Hierarchical (4)':   ['store_rolling_7','store_rolling_28',
                            'dept_rolling_7','dept_rolling_28'],
    'Categoricals (6)':   ['store_id_enc','item_id_enc','dept_id_enc',
                            'cat_id_enc','state_id_enc','weekday_enc'],
}
for group, cols in groups.items():
    print(f'  {group}')
print(f'\n  Total: {len(FEATURE_COLS)} features')

34 features confirmed across 6 groups. All features present in `df`.

**Expected nulls — XGBoost handles natively via surrogate splits:**
- `price_change_pct` / `price_drop` / `price_increase` / `price_rel_28`:
  490K–492K nulls — rows where `sell_price=0` (product not yet priced, ~1.7% of train)
- `lag_7`: 182K nulls — series start edges + gap boundary rows
- `lag_14`: 396K nulls — wider lookback crosses more gap boundaries
- `lag_28`: 823K nulls — furthest lookback, most gap boundaries crossed
- `lag_1`: 0 nulls — every training row has at least `lag_1` by construction
  (rows with no lag signal excluded from training via `has_any_lag` filter)

**Leakage check:** `lag_1 ↔ target` correlation = 0.531 — strong positive,
never 1.0. No leakage. ✓

## 13. Save Feature Matrix

Save as **Parquet** — not CSV. At this row count parquet is ~5x smaller
and ~10x faster to load than CSV. Preserves dtypes exactly (no
re-casting needed in notebook 05).

Three files:
- `features_train.parquet` — training rows, all features, target, meta
- `features_val.parquet` — validation rows (same columns)
- `feature_cols.pkl` — the exact `FEATURE_COLS` list, loaded by notebook 05
  so feature order is guaranteed consistent between train and inference

In [ ]:
import pickle

save_cols = META_COLS + FEATURE_COLS + [TARGET_COL]

train_df[save_cols].to_parquet(
    f'{PROCESSED_DIR}/features_train.parquet', index=False
)
val_df[save_cols].to_parquet(
    f'{PROCESSED_DIR}/features_val.parquet', index=False
)

# Save feature list — notebook 05 imports this directly
with open(f'{PROCESSED_DIR}/feature_cols.pkl', 'wb') as f:
    pickle.dump(FEATURE_COLS, f)

print('Saved:')
print(f'  features_train.parquet — {len(train_df):,} rows × {len(save_cols)} cols')
print(f'  features_val.parquet   — {len(val_df):,} rows × {len(save_cols)} cols')
print(f'  feature_cols.pkl       — {len(FEATURE_COLS)} features')
print(f'  label_encoders.pkl     — {len(label_encoders)} encoders')
print()
print(f'Train file size: {os.path.getsize(f"{PROCESSED_DIR}/features_train.parquet") / 1e6:.1f} MB')
print(f'Val file size:   {os.path.getsize(f"{PROCESSED_DIR}/features_val.parquet") / 1e6:.1f} MB')

## 14. Feature Distribution Spot-Check

Visual sanity check on the most important feature groups before
handing off to XGBoost. Purpose: catch any engineering bugs
(e.g. all-zero columns, impossible values, leakage artefacts)
before training.

In [ ]:
# ── Lag feature distributions ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, lag in zip(axes, LAG_DAYS):
    col = f'lag_{lag}'
    vals = train_df[col].clip(upper=10)
    ax.hist(vals, bins=30, color='steelblue', edgecolor='none')
    ax.set_title(f'{col}\nmean={train_df[col].mean():.2f}  null={train_df[col].isna().mean()*100:.1f}%')
    ax.set_xlabel('units (clipped at 10)')
plt.suptitle('Lag Feature Distributions (train set, clipped at 10 for display)', y=1.01)
plt.tight_layout()
plt.show()

# ── SNAP and event flag rates ─────────────────────────────────────────────
print('Flag feature rates (train set):')
flag_cols = ['is_event', 'is_closed_holiday', 'is_pre_closed_holiday',
             'is_snap', 'is_weekend', 'price_drop', 'price_increase']
for col in flag_cols:
    rate = train_df[col].mean() * 100
    print(f'  {col:<28} {rate:.1f}% of rows are 1')

# ── Price change distribution ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
nonzero_pct = train_df[train_df['price_change_pct'] != 0]['price_change_pct']
nonzero_pct.clip(lower=-30, upper=30).hist(bins=60, ax=ax, color='steelblue', edgecolor='none')
ax.axvline(0, color='red', linestyle='--', linewidth=1.5)
ax.set_title('price_change_pct (non-zero rows, clipped ±30%)')
ax.set_xlabel('% change')
plt.tight_layout()
plt.show()

print(f'\nRows with price change:    {(train_df["price_change_pct"] != 0).sum():,} '
      f'({(train_df["price_change_pct"] != 0).mean()*100:.1f}%)')
print(f'Rows with price drop:      {train_df["price_drop"].sum():,}')
print(f'Rows with price increase:  {train_df["price_increase"].sum():,}')

# ── Target distribution ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_df['target'], bins=50, color='steelblue', edgecolor='none')
axes[0].set_title(f'target (log1p)  skew={train_df["target"].skew():.2f}')
axes[1].hist(train_df['target'][train_df['target'] > 0], bins=50, color='steelblue', edgecolor='none')
axes[1].set_title('target (log1p, non-zero only)')
for ax in axes:
    ax.set_xlabel('log1p(units_sold)')
plt.suptitle('Target Distribution — Train Set', y=1.01)
plt.tight_layout()
plt.show()

# ── Gap flag spot-check on representative series ──────────────────────────
rep = train_df[train_df['id'] == REP_SERIES].sort_values('date')
print(f'\nRepresentative series ({REP_SERIES}):')
print(f'  Rows:        {len(rep):,}')
print(f'  Gap rows:    {rep["lag_1"].isna().sum():,} ({rep["lag_1"].isna().mean()*100:.1f}%)')
print(f'  lag_1 range: {rep["lag_1"].min():.0f} – {rep["lag_1"].max():.0f}')
print(f'  price range: ${rep["sell_price"].min():.2f} – ${rep["sell_price"].max():.2f}')

### Spot-check: Flag feature rates and price change distribution

**Flag features** are binary (0/1) columns. The rates below confirm each
feature fired at the expected frequency based on EDA findings.

| Feature | Rate | Expected | Notes |
|---|---|---|---|
| `is_event` | 7.9% | ~8% | ~1 event per 12 days in the M5 calendar ✓ |
| `is_closed_holiday` | 0.6% | ~0.5% | Christmas + Thanksgiving only — ~10 days/year ✓ |
| `is_pre_closed_holiday` | 2.2% | ~2% | 3 lead days × ~10 holidays/year ✓ |
| `is_snap` | 32.7% | ~33% | SNAP distributed ~1 in 3 days per state ✓ |
| `is_weekend` | 28.6% | ~29% | 2 of 7 days ✓ |
| `price_drop` | 0.5% | low | Price changes are infrequent — weekly price records mean most days show no change ✓ |
| `price_increase` | 0.8% | low | Slightly more increases than drops — consistent with gradual inflation over 5 years ✓ |

**Price changes** affect only 3.0% of rows — expected, since prices are
recorded weekly and most products hold their price for multiple weeks at a
time. Of the rows with a price change, increases slightly outnumber drops
(222K vs 153K). The asymmetric demand response to these events (r=0.553
for drops vs r=0.180 for increases at product-store level, EDA Section 18)
is why `price_drop` and `price_increase` are encoded as separate features
rather than a single signed flag.

**Representative series** (`FOODS_3_163_CA_3_validation`): 1,444 training
rows, zero gap rows, `lag_1` range 0–12 units, price stable at $2.00
throughout the training period. No anomalies — this series is clean and
will serve as a reliable spot-check series in notebook 05.

## 15. Feature Engineering Summary

### What was built and why

| Group | Count | Features | EDA Justification |
|---|---|---|---|
| Temporal | 7 | `day_of_week`, `day_of_month`, `week_of_year`, `month_num`, `is_weekend`, `is_month_start`, `is_month_end` | Calendar cycles visible in seasonal decomposition |
| Event/SNAP | 5 | `is_event`, `is_closed_holiday`, `days_to_closed_holiday`, `is_pre_closed_holiday`, `is_snap` | SNAP: +10–32% FOODS uplift by state. Christmas/Thanksgiving demand lives in lead days, not day-of |
| Price | 5 | `sell_price`, `price_change_pct`, `price_drop`, `price_increase`, `price_rel_28` | Asymmetric elasticity confirmed at product-store level: r=0.553 drops vs r=0.180 increases (EDA Section 18) |
| Lag/Rolling | 7 | `lag_1`, `lag_7`, `lag_14`, `lag_28`, `rolling_mean_7`, `rolling_mean_28`, `rolling_std_7` | SARIMA AR(1) p≈0.000 — recent demand is the strongest predictor |
| Hierarchical | 4 | `store_rolling_7`, `store_rolling_28`, `dept_rolling_7`, `dept_rolling_28` | Stores vary 3x in size; departments correlated 0.64–0.96 but not perfectly |
| Categoricals | 6 | `store_id_enc`, `item_id_enc`, `dept_id_enc`, `cat_id_enc`, `state_id_enc`, `weekday_enc` | XGBoost needs integer-encoded IDs to learn per-store/dept demand baselines |
| **Total** | **34** | | |

### What was deliberately excluded

| Excluded | Reason |
|---|---|
| Department-level price features | r < 0.13 at department level — no signal (EDA Section 18) |
| Global SNAP flag | State-specific uplift rates differ 3x — replaced by state-aware `is_snap` |
| Raw `event_name_1` string | Too many categories; `is_event` + `is_closed_holiday` + lead windows capture the relevant signal |
| Lags beyond 28 days | Matches the 28-day forecast horizon — longer lags add noise, not signal |
| `price_change_pct_raw` | Preserved in meta columns for anomaly detection in notebook 07, not a model feature |

### Outputs

| File | Contents | Used in |
|---|---|---|
| `features_train.parquet` | 28.7M rows × 39 cols — 34 features + target + 4 meta | Notebook 05 XGBoost training |
| `features_val.parquet` | 11.1M rows × 39 cols — same schema | Notebook 05 evaluation |
| `feature_cols.pkl` | Ordered list of 34 feature column names | Notebook 05 + app layer |
| `label_encoders.pkl` | 6 LabelEncoder objects | App layer inverse-transform |

### Known limitations

- **Observed sales ≠ true demand.** All features derived from `units_sold`
  carry the same demand proxy caveat established in notebook 01 — zero sales
  on a given day may reflect a stockout (unmet demand) or genuine demand
  absence, indistinguishable without inventory data.
- **Gap-aware nulling covers structural gaps but not intermittent zeros.**
  Within an active window, a day with zero sales is treated as genuine zero
  demand. Some of these may be unrecorded stockouts that would bias lag
  features slightly downward for high-velocity products.
- **`rolling_std_7`** feeds directly into the safety stock formula in the
  Streamlit app (`safety_stock = z × rolling_std_7 × √lead_time`) — this
  approximates demand volatility from observed sales variance, which
  understates true demand volatility to the extent that stockouts suppress
  observed sales below true demand.